# 🎬 OpenShorts on Google Colab
Generator video pendek 9:16 otomatis dari video panjang / YouTube menggunakan AI.

**Fitur yang sudah aktif:**
- Custom LLM: `Gemini 3.8 Flash` via `https://my.ivyy.dpdns.org/v1`
- Whisper ASR untuk transkrip otomatis
- YOLOv8 + MediaPipe auto reframing
- Auto subtitle & hook
- Cloudflare Tunnel untuk akses web UI publik langsung dari browser Anda.

### Langkah 1: Cek GPU

In [ ]:
!nvidia-smi

import os
if not os.path.exists('app.py'):
    !git clone https://github.com/digitaldeveloperinfo/shrotclone.git /content/openshorts
    %cd /content/openshorts
else:
    !git fetch origin main && git reset --hard origin/main

!apt-get update -qq
!apt-get install -y -qq ffmpeg curl nodejs npm

In [ ]:
import os
if not os.path.exists('app.py'):
    !git clone https://github.com/digitaldeveloperinfo/shrotclone.git /content/openshorts
    %cd /content/openshorts
else:
    !git fetch origin main && git reset --hard origin/main

!apt-get update -qq
!apt-get install -y -qq ffmpeg curl nodejs npm

### Langkah 3: Install Dependensi Python & Frontend

In [ ]:
import sys
# Install Python requirements using active Python interpreter
!{sys.executable} -m pip install -r requirements.txt
!{sys.executable} -m pip install boto3 "mediapipe==0.10.14" "scenedetect>=0.6.4" py3langid faster-whisper ultralytics torch torchvision transnetv2-pytorch json-repair

# Install Dashboard dependencies
!cd dashboard && npm ci

### Langkah 4: Konfigurasi Environment (.env)
Secara default menggunakan endpoint Custom LLM Gemini 3.8 Flash yang sudah dikonfigurasi.

In [ ]:
env_content = '''
LLM_BASE_URL=https://my.ivyy.dpdns.org/v1
LLM_API_KEY=sk-5f238e76072d7926-8d3e86-14b7d8e3
LLM_MODEL=antigravity/gemini-3.8-flash-tiered
LLM_PROVIDER=openai
LLM_TIMEOUT=120
LLM_SCORE_BATCH=3
VITE_PROXY_TARGET=http://127.0.0.1:8000
QUALITY_GATE_MIN_HEIGHT=0
RATE_LIMIT_ENABLED=0
DEBUG_LOGS=true
'''
with open('.env', 'w') as f:
    f.write(env_content.strip() + '\n')
print('✅ File .env berhasil dibuat!')

### Langkah 5: Jalankan OpenShorts & Dapatkan Public URL
Sel ini akan menjalankan Backend (FastAPI), Frontend (Vite), dan Cloudflare Tunnel.
Tunggu beberapa detik hingga URL **https://xxxx.trycloudflare.com** muncul di output di bawah.

In [ ]:
import os, subprocess, time, urllib.request, re, sys

# 1. Bersihkan port dari proses lama jika sel di-restart
!pkill -9 -f uvicorn || true
!pkill -9 -f vite || true
!pkill -9 -f cloudflared || true
time.sleep(1)

# 1.5 Pastikan seluruh dependensi ML pipeline sudah terpasang di kernel aktif ini
required_ml = [
    ('scenedetect', 'scenedetect>=0.6.4'),
    ('faster_whisper', 'faster-whisper'),
    ('ultralytics', 'ultralytics'),
    ('mediapipe', 'mediapipe==0.10.14'),
    ('transnetv2_pytorch', 'transnetv2-pytorch'),
    ('json_repair', 'json-repair'),
    ('torch', 'torch torchvision'),
]
for mod, pkg in required_ml:
    try:
        __import__(mod)
    except ImportError:
        print(f'📦 Menginstal paket yang belum terpasang: {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(), check=False)

log_dir = '/content' if os.path.exists('/content') else os.getcwd()
backend_log_path = os.path.join(log_dir, 'backend.log')
frontend_log_path = os.path.join(log_dir, 'frontend.log')
tunnel_log_path = os.path.join(log_dir, 'tunnel.log')

# 2. Unduh Cloudflared jika belum ada
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('⬇️ Mengunduh Cloudflared tunnel...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/usr/local/bin/cloudflared'
    )
    os.chmod('/usr/local/bin/cloudflared', 0o755)

# 3. Jalankan Backend (FastAPI) via sys.executable
print('🚀 Menjalankan Backend API (:8000)...')
backend_log = open(backend_log_path, 'w', buffering=1, encoding='utf-8')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=backend_log,
    stderr=subprocess.STDOUT
)

# Verifikasi kesehatan backend
print('⏳ Menunggu backend siap...')
backend_ready = False
for _ in range(25):
    time.sleep(1)
    if backend_proc.poll() is not None:
        print('❌ Backend berhenti saat startup!')
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            if r.status == 200:
                backend_ready = True
                print('✅ Backend FastAPI aktif & sehat di port 8000!')
                break
    except Exception:
        pass

if not backend_ready:
    print('\n❌ Backend gagal dijalankan! Log output:')
    if os.path.exists(backend_log_path):
        with open(backend_log_path, 'r', errors='ignore') as f:
            print(f.read())
    raise RuntimeError('Backend gagal aktif. Periksa log di atas.')

# 4. Jalankan Frontend (Vite) ke port 5173
print('⚡ Menjalankan Frontend Vite (:5173)...')
frontend_env = dict(os.environ, VITE_PROXY_TARGET='http://127.0.0.1:8000')
frontend_log = open(frontend_log_path, 'w', buffering=1, encoding='utf-8')
frontend_proc = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173'],
    cwd='dashboard',
    env=frontend_env,
    stdout=frontend_log,
    stderr=subprocess.STDOUT
)

time.sleep(3)
if frontend_proc.poll() is not None:
    print('❌ Frontend Vite berhenti saat startup! Log:')
    if os.path.exists(frontend_log_path):
        with open(frontend_log_path, 'r', errors='ignore') as f:
            print(f.read())
    raise RuntimeError('Frontend gagal aktif.')
print('✅ Frontend Vite aktif & siap di port 5173!')

# 5. Jalankan Cloudflare Tunnel ke port 5173
print('🌐 Membuka Cloudflare Tunnel publik...')
tunnel_cmd = f'cloudflared tunnel --url http://127.0.0.1:5173 > {tunnel_log_path} 2>&1 &' 
subprocess.Popen(tunnel_cmd, shell=True)

url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists(tunnel_log_path):
        with open(tunnel_log_path, 'r', errors='ignore') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
            if match:
                url = match.group(0)
                break

if url:
    print('\n' + '='*65)
    print('🎉 OPENSHORTS BERHASIL AKTIF!')
    print(f'👉 Akses Dashboard di URL ini: {url}')
    print('='*65 + '\n')
    print('📌 Live backend logs di bawah ini:')
    print('-----------------------------------------------------------------')
else:
    print('⚠️ Gagal mendeteksi URL tunnel. Cek log:')
    if os.path.exists(tunnel_log_path):
        with open(tunnel_log_path) as f:
            print(f.read())

# Streaming log backend secara realtime ke output sel
try:
    log_pos = 0
    while True:
        time.sleep(1.5)
        if os.path.exists(backend_log_path):
            with open(backend_log_path, 'r', errors='ignore') as f:
                f.seek(log_pos)
                new_data = f.read()
                log_pos = f.tell()
                if new_data:
                    print(new_data, end='', flush=True)
        if backend_proc.poll() is not None:
            print('\n❌ Backend berhenti tak terduga!')
            break
        if frontend_proc.poll() is not None:
            print('\n❌ Frontend berhenti tak terduga!')
            break
except KeyboardInterrupt:
    print('\n🛑 Mematikan server...')
    !pkill -9 -f uvicorn || true
    !pkill -9 -f vite || true
    !pkill -9 -f cloudflared || true
